In [1]:
from datasets import load_dataset
from collections import defaultdict
import numpy as np

data = load_dataset("lhoestq/conll2003")
label_names = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]


def shape_features(tokens, i):
    w = tokens[i]
    return {
        "is_title": w.istitle(),
        "is_upper": w.isupper(),
        "has_digit": any(c.isdigit() for c in w),
        "is_first": i == 0,
    }


def baseline_features(tokens, i):
    f = shape_features(tokens, i)
    f["word"] = tokens[i].lower()
    f["prev_word"] = tokens[i - 1].lower() if i > 0 else "<START>"
    f["next_word"] = tokens[i + 1].lower() if i < len(tokens) - 1 else "<END>"
    return f


def build(split, feature_fn):
    X, y = [], []
    for row in data[split]:
        tokens = row["tokens"]
        for i in range(len(tokens)):
            X.append(feature_fn(tokens, i))
        y.extend(label_names[t] for t in row["ner_tags"])
    return X, y


def to_sentences(flat, split):
    out, k = [], 0
    for row in data[split]:
        n = len(row["tokens"])
        out.append(list(flat[k:k + n]))
        k += n
    return out


def get_entities(tags):
    ents, start, label = set(), None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if t == "O" or t.startswith("B-") or (t.startswith("I-") and t[2:] != label):
            if label is not None:
                ents.add((label, start, i))
            label, start = (t[2:], i) if t != "O" else (None, None)
    return ents


def report(true_sents, pred_sents):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for t, p in zip(true_sents, pred_sents):
        T, P = get_entities(t), get_entities(p)
        for e in T & P:
            tp[e[0]] += 1
        for e in P - T:
            fp[e[0]] += 1
        for e in T - P:
            fn[e[0]] += 1

    print(f"{'type':8}{'precision':>10}{'recall':>10}{'f1':>10}{'support':>10}")
    for lab in sorted(set(tp) | set(fp) | set(fn)) + ["ALL"]:
        if lab == "ALL":
            a, b, c = sum(tp.values()), sum(fp.values()), sum(fn.values())
        else:
            a, b, c = tp[lab], fp[lab], fn[lab]
        p = a / (a + b) if a + b else 0
        r = a / (a + c) if a + c else 0
        f = 2 * p * r / (p + r) if p + r else 0
        print(f"{lab:8}{p:10.3f}{r:10.3f}{f:10.3f}{a + c:10d}")

dataset_infos.json:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.07MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  281kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  259kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [2]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

X_train, y_train = build("train", baseline_features)
X_test, y_test = build("test", baseline_features)

vec = DictVectorizer()
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)

clf = LogisticRegression(max_iter=100)
clf.fit(Xtr, y_train)
pred = clf.predict(Xte)

true_sents = to_sentences(y_test, "test")
pred_sents = to_sentences(pred, "test")
print("BASELINE (no embeddings)")
report(true_sents, pred_sents)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


BASELINE (no embeddings)
type     precision    recall        f1   support
LOC          0.793     0.740     0.766      1668
MISC         0.728     0.652     0.688       702
ORG          0.605     0.606     0.605      1661
PER          0.649     0.785     0.710      1617
ALL          0.683     0.703     0.693      5648


In [3]:
!pip install -q gensim

import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-50")
DIM = 50
zero = np.zeros(DIM)


def vec_of(word):
    w = word.lower()
    return glove[w] if w in glove else zero


def embed_features(split):
    shapes, dense = [], []
    for row in data[split]:
        tokens = row["tokens"]
        for i in range(len(tokens)):
            shapes.append(shape_features(tokens, i))
            prev_v = vec_of(tokens[i - 1]) if i > 0 else zero
            next_v = vec_of(tokens[i + 1]) if i < len(tokens) - 1 else zero
            dense.append(np.concatenate([vec_of(tokens[i]), prev_v, next_v]))
    return shapes, np.array(dense)


shapes_tr, dense_tr = embed_features("train")
shapes_te, dense_te = embed_features("test")
print(dense_tr.shape, dense_te.shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 88.4 MB/s eta 0:00:00
[==================================================] 100.0% 66.0/66.0MB downloaded
(203621, 150) (46435, 150)


In [4]:
from scipy.sparse import csr_matrix, hstack


shape_vec = DictVectorizer()
S_tr = shape_vec.fit_transform(shapes_tr)
S_te = shape_vec.transform(shapes_te)

Xtr_e = hstack([S_tr, csr_matrix(dense_tr)]).tocsr()
Xte_e = hstack([S_te, csr_matrix(dense_te)]).tocsr()

clf_e = LogisticRegression(max_iter=100)
clf_e.fit(Xtr_e, y_train)
pred_e = clf_e.predict(Xte_e)

print("WITH GLOVE EMBEDDINGS")
report(to_sentences(y_test, "test"), to_sentences(pred_e, "test"))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


WITH GLOVE EMBEDDINGS
type     precision    recall        f1   support
LOC          0.721     0.794     0.756      1668
MISC         0.567     0.638     0.601       702
ORG          0.565     0.633     0.597      1661
PER          0.803     0.843     0.823      1617
ALL          0.677     0.742     0.708      5648
